# 00 · Start here

These notebooks rebuild the study's **whole model library** from the EDHS 2024-25 Individual
Recode. The study's own models, three models on the 21 predictors, are rebuilt far more quickly
by `reproduction/`; this folder rebuilds everything the study tried on the way:

- the cohort, the outcome ladder and the 21 predictors (notebook 01, the same as `reproduction/`);
- the **candidate pool**: every one of the survey's 6,648 columns screened, 95 admitted (02);
- the held-out women and the development folds (03);
- the **four predictor sets**: the clean pool of 86 (after the near-duplicate analysis), the 21,
  the short set and the parsimonious set (04-06);
- **eleven model families** tuned on every set at every rung, each with an **isotonic** and a
  **Platt** recalibration, and **stacked** and **cumulative-link** ensembles (07);
- **CatBoost** and two **neural networks** on a GPU (08);
- the ensembles of the three- and four-model lineups, and the study's own grid-tuned models (09);
- every result checked against the study's (10).

Run them in order; each reads only what the earlier ones wrote and ends by checking its results
against the study's (in `reference/`). What all of it found is written up in `research_record/`.

## Before you start

1. **The data.** Obtain `ETIR8AFL.DTA` (Ethiopia DHS 2024-25, Individual Recode, Stata) from
   The DHS Program and put it in `data/`. It is licensed to you and must not be shared;
   nothing in this folder contains survey rows, and everything the notebooks write stays in
   `outputs/`, `results/`, `study/` and `quick/`, which are never committed.
2. **The environment.** Python 3.12 or 3.13, then `pip install -r requirements.txt`; on the GPU
   machine also `catboost` and `torch` (see the file).
3. **The time.** This is days of computing, not hours. The study ran it on this PC's 8 cores and
   a Colab T4 GPU over several days. Every model is saved as it finishes, and a notebook that is
   run again skips what is done, so it can be stopped and resumed at any point.

| Notebook | Does | Time |
|---|---|---|
| 01 | the cohort, the outcome ladder and the 21 predictors | 2 min |
| 02 | the candidate pool: every survey column screened, 95 admitted | 3 min |
| 03 | the held-out split and the development folds | under 1 min |
| 04 | a helper model for the selection steps (LightGBM on the 95) | 15 min per rung |
| 05 | near-duplicate predictors, and the clean pool of 86 | 5 min |
| 06 | the four predictor sets | 6 to 8 h |
| 07 | eleven families x four sets x three rungs, recalibrated, ensembled (CPU) | 3 to 5 days |
| 08 | CatBoost and the two networks (GPU) | about a day on a T4 |
| 09 | the lineup ensembles, and the study's grid-tuned models | 2 h |
| 10 | every result against the study's | 5 min |

To run them without opening Jupyter, from this folder:

```
python scripts/run_notebooks.py --only 01 02 03 04 05 06 07   # this machine
python scripts/run_notebooks.py --only 08                     # on the GPU machine
python scripts/run_notebooks.py --only 09 10
python scripts/run_notebooks.py --quick                       # a smoke run: 3 search trials per model
```

## Two switches, at the top of every notebook

- `QUICK = True` runs everything with three search trials per model. The numbers will not
  match the study's; use it to check that everything runs.
- `FOLLOW_STUDY = True` (the default): where a re-derived choice (a selection step, a helper
  model's settings) differs from the study's, the notebook says so and carries the study's
  choice forward, so that the later notebooks reproduce the study's numbers exactly.

## Folders

```
data/            put ETIR8AFL.DTA here
notebooks/       00-10
src/anc8/        the analysis package every notebook uses
scripts/         the stage scripts the notebooks run (train_library.py builds the library)
reference/       the study's results, for the checks
outputs/         what the notebooks write (analysis frames, tables)
results/         the library: one file per model, set and rung
study/           the study's own grid-tuned models
```

In [ ]:
# Is the environment the one the study used, and is the survey file in place?
import importlib, sys
from pathlib import Path

here = Path.cwd()
project = here.parent if here.name.startswith("notebooks") else here
want = {}
for line in (project / "requirements.txt").read_text().splitlines():
    if "==" in line and not line.startswith("#"):
        name, v = line.split("==")
        want[name.strip()] = v.strip()

MODULE = {"scikit-learn": "sklearn", "python-docx": "docx"}
print(f"python {sys.version.split()[0]}")
for name, v in want.items():
    try:
        m = importlib.import_module(MODULE.get(name, name))
        got = getattr(m, "__version__", "?")
        print(f"  {name:14s} {got:10s} {'' if got == v else f'(the study used {v})'}")
    except ImportError:
        print(f"  {name:14s} MISSING    (pip install -r requirements.txt)")

dta = [p for p in (project / "data").glob("*") if p.name.lower() == "etir8afl.dta"]
print("\nsurvey file:", dta[0] if dta else "NOT FOUND: put ETIR8AFL.DTA in data/")